# 9장. 사전학습: 이야기를 읽으며 배우기

8장의 빠른 GPT를 이야기 20만 편으로 학습해요. **T4 GPU가 필요하고, 30분 정도 걸립니다.** (런타임 유형: T4 GPU)

**이 장이 끝나면**: 드라이브에 `pretrain.pt`(학습된 모델)와 단계별 체크포인트가 저장됩니다.
GPU를 받지 못했거나 시간이 없다면 이 장은 읽기만 해도 돼요. 10장은 드라이브에 `pretrain.pt`가 없으면 저자의 체크포인트를 내려받습니다.


### 준비
드라이브 연결 창이 뜨면 허용하세요.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from modu_llm import Vocab, GPTConfig, GPT, count_params, generate
from modu_llm import load_stories, encode_texts, split_data, get_batch
from modu_llm import save_checkpoint, load_checkpoint, train, mount_drive

torch.manual_seed(42)
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("장치:", device)
if device == "cpu":
    print("⚠ GPU가 연결되지 않았어요. 런타임 → 런타임 유형 변경 → T4 GPU를 고르세요.")
    print("  GPU를 받을 수 없다면 이 장의 학습은 건너뛰고, 10장에서 저자 체크포인트를 쓰면 돼요.")
vocab = Vocab.build()
folder = mount_drive()


In [ ]:
# 학습 설정
N_STORIES = 200_000      # 학습에 쓸 이야기 수
TRAIN_MINUTES = 30       # 이번 실행에서 학습할 시간(분)
LR = 5e-4                # 학습률: 한 번에 가중치를 고치는 정도
BATCH = 32
EVAL_EVERY = 250         # 몇 단계마다 손실을 확인할지
SAVE_EVERY_MIN = 5       # 몇 분마다 저장할지
SNAPSHOTS = [250, 1000, 3000]   # 10장에서 비교할 단계별 체크포인트
ckpt_path = f"{folder}/pretrain.pt"


## 9.1 학습 데이터 준비: 메모리를 아끼는 인코딩

이야기 20만 편은 약 7천만 글자예요. 2장처럼 한 번에 번호 리스트로 바꾸면 파이썬 정수 수천만 개가 생겨 Colab 메모리가 부족해질 수 있습니다.
그래서 **이야기마다** 인코딩해 작은 정수 배열(`uint16`, 숫자 하나에 2바이트)로 이어 붙여요. 한 번 만든 배열은 드라이브에 저장해 두고 다음부터는 불러오기만 합니다.

**핵심 코드**: `data = encode_texts(stories, vocab)`


In [ ]:
tokens_path = f"{folder}/tokens_{N_STORIES}.npy"
if os.path.exists(tokens_path):
    data = np.load(tokens_path)
    print("저장된 데이터를 불러왔습니다.")
else:
    t0 = time.time()
    stories = load_stories(N_STORIES)          # 몇 분 걸립니다
    data = encode_texts(stories, vocab)
    np.save(tokens_path, data)
    seconds = time.time() - t0
    print(f"이야기 {len(stories):,}편 인코딩 완료 ({seconds:.0f}초)")

n_chars = len(data)
mb_array = round(data.nbytes / 1e6)
mb_list = round(n_chars * 36 / 1e6)
print(f"글자 수: {n_chars:,}")
print(f"uint16 배열: {mb_array:,}MB")
print(f"파이썬 정수 리스트였다면 약 {mb_list:,}MB")
train_data, val_data = split_data(data)


## 9.2 학습 한 단계: 예측하고, 틀린 만큼 고치기

학습 한 단계는 네 줄로 요약됩니다.

1. 배치를 꺼내 **예측**하고 손실을 계산한다 (`model(x, y)`)
2. 지난 기울기를 지운다 (`zero_grad`)
3. 손실을 줄이려면 가중치를 어느 방향으로 고칠지 계산한다 (`backward`)
4. 실제로 고친다 (`step`)

아래 셀은 다섯 단계만 실행해 손실이 줄어드는지 확인해요.

**핵심 코드**: `scaler.scale(loss).backward()` → `scaler.step(optimizer)`


In [ ]:
model = GPT(GPTConfig()).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def train_step():
    x, y = get_batch(train_data, model.cfg.block_size, BATCH, device)
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        _, loss = model(x, y)                  # ① 예측과 손실
    optimizer.zero_grad(set_to_none=True)      # ② 지난 기울기 지우기
    scaler.scale(loss).backward()              # ③ 고칠 방향 계산
    scaler.step(optimizer)                     # ④ 가중치 고치기
    scaler.update()
    return loss.item()

n_params = count_params(model)
print(f"파라미터 수: {n_params:,}")
for i in range(5):
    loss = train_step()
    print(f"단계 {i + 1}: 손실 {loss:.3f}")


## 9.3 구글 드라이브에 체크포인트 저장하기

Colab 세션은 언제든 끊길 수 있어요. 학습 중간 상태를 파일로 남기는 것을 **체크포인트**라고 합니다.
모델 가중치뿐 아니라 옵티마이저 상태, 지금까지의 단계 수, 손실 기록, 모델 설정까지 함께 저장해야 이어서 학습할 수 있어요.
연습용 파일로 저장하고 다시 불러와 봅니다.

**핵심 코드**: `save_checkpoint(경로, model, optimizer, step, history)`


In [ ]:
test_path = f"{folder}/test_ckpt.pt"
save_checkpoint(test_path, model, optimizer, step=5, history=[])

ckpt = torch.load(test_path, map_location="cpu")
print("체크포인트에 든 것:", list(ckpt.keys()))
print("모델 설정:", ckpt["cfg"])
size_mb = round(os.path.getsize(test_path) / 1e6)
print(f"파일 크기: {size_mb}MB")
os.remove(test_path)


## 9.4 끊겨도 이어서 학습하기

본 학습입니다. 드라이브에 `pretrain.pt`가 있으면 **그 상태에서 이어서**, 없으면 처음부터 학습해요.
학습은 패키지의 `train` 함수가 맡습니다. 9.2의 학습 한 단계를 `TRAIN_MINUTES`분 동안 되풀이하면서, 손실을 기록하고 정해진 때마다 저장해요.
세션이 끊기면 런타임을 다시 연결하고 **준비 셀부터 다시 실행**하세요. 저장된 지점부터 이어집니다.

**핵심 코드**: `if os.path.exists(ckpt_path): ... load_checkpoint(...)`


In [ ]:
if os.path.exists(ckpt_path):
    model, optimizer, step, history = load_checkpoint(ckpt_path, device, optimizer_lr=LR)
    print(f"{step}단계에서 이어서 학습합니다.")
else:
    model = GPT(GPTConfig()).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    step = 0
    history = []
    print("처음부터 학습합니다.")

step, history = train(
    model, optimizer, train_data, val_data,
    minutes=TRAIN_MINUTES,
    ckpt_path=ckpt_path,
    batch_size=BATCH,
    eval_every=EVAL_EVERY,
    save_every_min=SAVE_EVERY_MIN,
    start_step=step,
    history=history,
    snapshots=SNAPSHOTS,
    vocab=vocab,
)

last = history[-1]
print(f"학습 끝: {step}단계 | 검증 손실 {last['val']:.3f} | 저장: {ckpt_path}")


## 9.5 손실 그래프 읽기와 과적합

학습 손실과 검증 손실이 **함께** 내려가면 처음 보는 이야기에도 잘 통하도록 배우고 있다는 뜻이에요.
학습 손실만 내려가고 검증 손실이 올라가기 시작하면 **과적합**, 즉 학습한 글을 외우기 시작했다는 신호입니다.


In [ ]:
steps = []
train_losses = []
val_losses = []
for h in history:
    steps.append(h["step"])
    train_losses.append(h["train"])
    val_losses.append(h["val"])

plt.plot(steps, train_losses, label="train")
plt.plot(steps, val_losses, label="validation")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

여러 시작 문장으로 이야기를 만들어 보세요. 1장의 2분짜리 모델과 비교하면 어떤가요?


In [ ]:
for prompt in ["어느 날, ", "밤하늘에 ", "엄마가 "]:
    story = generate(model, vocab, prompt, max_new_tokens=200)
    print(prompt + story)
    print("-" * 40)


## 실제 LLM에서는

- 실제 LLM의 사전학습은 수천 개의 GPU로 수조 개의 토큰을 몇 주에서 몇 달 동안 학습합니다.
- 학습률을 처음에 천천히 올렸다가 점점 낮추는 일정(warmup, cosine)을 쓰고, 웹 문서·책·코드 등 여러 데이터를 비율을 정해 섞어요.
